# MixMIL with AnnData: multi-label prediction

This tutorial predicts several independent binary phenotypes for each donor. The cell embeddings live in `.obsm`, donor metadata and labels live in `.obs`, and the adapter returns one target column per label.

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
import torch

from mixmil import MixMIL
from mixmil.anndata import (
    add_attention_to_anndata,
    add_bag_predictions_to_anndata,
    prepare_anndata,
)

rng = np.random.default_rng(7)
n_bags, cells_per_bag, n_features, n_labels = 48, 20, 8, 2
n_obs = n_bags * cells_per_bag
embedding = rng.normal(size=(n_obs, n_features)).astype('float32')
bag_embedding = embedding.reshape(n_bags, cells_per_bag, n_features).mean(axis=1)
coefficients = rng.normal(size=(n_features, n_labels))
label_logits = bag_embedding @ coefficients
bag_labels = (1 / (1 + np.exp(-label_logits)) > 0.5).astype('float32')
bag_ids = np.repeat([f'donor_{i:03d}' for i in range(n_bags)], cells_per_bag)
split = np.repeat(np.where(np.arange(n_bags) < 36, 'train', 'test'), cells_per_bag)
age = np.repeat(rng.integers(20, 80, size=n_bags), cells_per_bag)
sex = np.repeat(rng.integers(0, 2, size=n_bags), cells_per_bag)
obs = pd.DataFrame({
    'donor': bag_ids,
    'split': split,
    'age': age,
    'sex': sex,
    'label_a': np.repeat(bag_labels[:, 0], cells_per_bag),
    'label_b': np.repeat(bag_labels[:, 1], cells_per_bag),
})
adata = ad.AnnData(X=np.zeros((n_obs, 1), dtype='float32'), obs=obs)
adata.obsm['X_mixmil'] = embedding
adata

## Convert AnnData to donor bags

A sequence of target columns creates a multi-output `Y` tensor with shape `(n_bags, n_labels)`.

In [ ]:
prepared = prepare_anndata(
    adata,
    bag_key='donor',
    target_key=['label_a', 'label_b'],
    feature_key='X_mixmil',
    fixed_effects=['sex', 'age'],
    split_key='split',
    scale_fixed_effects=['age'],
)

prepared.train.Y.shape, prepared.test.Y.shape, prepared.fixed_effect_names

In [ ]:
model = MixMIL(
    Q=prepared.n_features,
    K=prepared.n_fixed_effects,
    P=2,
    likelihood='binomial',
    n_trials=1,
    mean_field=True,
)
history = model.fit(
    prepared.train.Xs, prepared.train.F, prepared.train.Y,
    n_epochs=150, batch_size=16, verbose=False,
)
logits = model.predict_logits(prepared.test.Xs, prepared.test.F)
probabilities = torch.sigmoid(logits)
predicted_labels = (probabilities >= 0.5).to(torch.int64)
print('test logits:', logits.shape)
print('test probabilities:', probabilities[:5])

## Put results back into AnnData

Attention is cell-level and is stored in `.obs`; donor predictions remain donor-level in `.uns`.

In [ ]:
result = add_attention_to_anndata(
    adata, prepared.test, model.get_weights(prepared.test.Xs)[0],
    key='mixmil_attention',
)
result = add_bag_predictions_to_anndata(
    result, prepared.test.bag_ids, logits, key='mixmil', inplace=True,
)
print(result.obs.filter(like='mixmil_attention').head())
print(result.uns['mixmil']['bag_ids'][:3])